In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
!pip install pytorch-lightning timm torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 67.3 MB/s eta 0:00:00


In [4]:
%%writefile /content/drive/MyDrive/XAI_Cloud_Run/train.py
import os
import torch
import pandas as pd
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms
from sklearn.model_selection import train_test_split

from config import Config
from data_loaders import Derm7ptDataset
from models import GatedHybridCBM
from engine import CBM_System

def main():
    # 1. Cloud Reproducibility Lock
    pl.seed_everything(42)
    print("Initializing Cloud Training Sequence...")

    # 2. Hardcoded Linux Drive Paths
    DATA_DIR = "/content/drive/MyDrive/XAI_Cloud_Run/derm7pt"
    META_CSV = os.path.join(DATA_DIR, "meta", "meta.csv")
    IMAGE_DIR = os.path.join(DATA_DIR, "images")
    CHECKPOINT_DIR = "/content/drive/MyDrive/XAI_Cloud_Run/cloud_checkpoints/"

    full_df = pd.read_csv(META_CSV)

    # 3. Stratification Fix: Convert all rare conditions to binary 1/0
    binary_labels = (full_df['diagnosis'].str.lower() == 'melanoma').astype(int)

    train_df, val_df = train_test_split(
        full_df, test_size=0.2, stratify=binary_labels, random_state=42
    )

    # 4. Mode Collapse Fix: Weighted Sampler
    print("\nCalculating Class Weights...")
    is_melanoma = train_df['diagnosis'].str.lower() == 'melanoma'
    count_melanoma = is_melanoma.sum()
    count_other = len(train_df) - count_melanoma

    weight_melanoma = 1.0 / count_melanoma if count_melanoma > 0 else 1.0
    weight_other = 1.0 / count_other if count_other > 0 else 1.0

    sample_weights = [weight_melanoma if label == 'melanoma' else weight_other
                      for label in train_df['diagnosis'].str.lower()]

    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True
    )

    basic_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor()
    ])

    train_dataset = Derm7ptDataset(metadata_df=train_df, image_dir=IMAGE_DIR, transforms=basic_transform)
    val_dataset = Derm7ptDataset(metadata_df=val_df, image_dir=IMAGE_DIR, transforms=basic_transform)

    # Shuffle MUST be False when using a sampler
    train_loader = DataLoader(train_dataset, batch_size=8, sampler=sampler, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)

    print("\nBuilding GatedHybridCBM Architecture...")
    model = GatedHybridCBM(
        concept_dims=Config.get_active_concept_dims(),
        backbone=Config.BACKBONE,
        feat_proj_dim=Config.FEAT_PROJ_DIM,
        hidden=Config.HIDDEN_DIM,
        dropout=Config.DROPOUT
    )
    lightning_system = CBM_System(model, Config)

    # 5. Cloud Failsafes (Saving directly to Drive)
    checkpoint_callback = ModelCheckpoint(
        dirpath=CHECKPOINT_DIR,
        filename="gated-cbm-{epoch:02d}-{val_auc:.3f}",
        monitor="val_auc",
        mode="max",
        save_top_k=1,
    )

    early_stop_callback = EarlyStopping(
        monitor="val_auc",
        min_delta=0.005,
        patience=7,
        verbose=True,
        mode="max"
    )

    logger = TensorBoardLogger("/content/drive/MyDrive/XAI_Cloud_Run/tb_logs", name="GatedHybridCBM_Derm7pt")

    trainer = pl.Trainer(
        max_epochs=50,
        accelerator="auto",
        devices="auto",
        logger=logger,
        callbacks=[checkpoint_callback, early_stop_callback],
        precision="16-mixed",
        log_every_n_steps=10
    )

    print("\nLaunching Training Sequence...")
    trainer.fit(lightning_system, train_dataloaders=train_loader, val_dataloaders=val_loader)
    print(f"Training Complete. Best model saved at: {checkpoint_callback.best_model_path}")

if __name__ == "__main__":
    main()

Overwriting /content/drive/MyDrive/XAI_Cloud_Run/train.py


In [6]:
file_path = '/content/drive/MyDrive/XAI_Cloud_Run/train.py'

with open(file_path, 'r') as f:
    text = f.read()

# Rip out the strict stratify parameter
text = text.replace("stratify=binary_labels, ", "")

with open(file_path, 'w') as f:
    f.write(text)

print("OVERRIDE SUCCESSFUL: Stratification constraint destroyed.")

OVERRIDE SUCCESSFUL: Stratification constraint destroyed.


In [8]:
%%writefile /content/drive/MyDrive/XAI_Cloud_Run/data_loaders.py
import os
import pandas as pd
import numpy as np
import torch
from PIL import Image

from base_dataset import BaseConceptDataset

class Derm7ptDataset(BaseConceptDataset):
    def __init__(self, metadata_df, image_dir, transforms=None):
        super().__init__(metadata_df, transforms)
        self.image_dir = image_dir
        self.concept_cols = [
            "pigment_network", "streaks", "pigmentation",
            "regression_structures", "dots_and_globules",
            "blue_whitish_veil", "vascular_structures"
        ]
        self._prepare_samples()

    def _parse_concept(self, val, col_idx):
        if pd.isna(val) or val == -1 or str(val).strip() == '-1' or str(val).lower() == 'unknown':
            return 0, 0.0

        if isinstance(val, (int, float)):
            return int(val), 1.0

        val_str = str(val).lower().strip()

        if val_str in ['absent', 'none', 'false', '0']:
            return 0, 1.0

        if val_str in ['present', 'typical', 'regular', 'true', '1']:
            return 1, 1.0

        if val_str in ['atypical', 'irregular', '2']:
            if col_idx == 5:
                return 2, 1.0
            else:
                return 1, 1.0

        return 1, 1.0

    def _prepare_samples(self):
        for _, row in self.df.iterrows():
            # FIX: Dynamically check the correct Derm7pt image columns
            img_file = row.get('derm', row.get('clinic', row.get('image_path', '')))
            img_path = os.path.join(self.image_dir, str(img_file))

            concepts, masks = [], []
            for i, col in enumerate(self.concept_cols):
                val = row.get(col, np.nan)
                c_val, c_mask = self._parse_concept(val, i)
                concepts.append(c_val)
                masks.append(c_mask)

            label = 1 if str(row.get('diagnosis')).lower() == 'melanoma' else 0
            self.samples.append((img_path, concepts, masks, label))

    def __getitem__(self, idx):
        img_path, concepts, masks, label = self.samples[idx]
        try:
            # FIX: Prevent directory crashes if an image is missing
            if os.path.isdir(img_path):
                raise IsADirectoryError()
            img = Image.open(img_path).convert("RGB")
        except (FileNotFoundError, IsADirectoryError):
            img = Image.new('RGB', (224, 224))

        if self.transforms:
            img = self.transforms(img)

        return (
            img,
            torch.tensor(concepts, dtype=torch.long),
            torch.tensor(masks, dtype=torch.float32),
            torch.tensor(label, dtype=torch.float32)
        )

class HAM10000Dataset(BaseConceptDataset):
    def __init__(self, metadata_df, image_paths_dict, transforms=None):
        super().__init__(metadata_df, transforms)
        self.image_paths_dict = image_paths_dict
        self._prepare_samples()

    def _prepare_samples(self):
        for _, row in self.df.iterrows():
            img_id = row['image_id']
            img_path = self.image_paths_dict.get(img_id)
            if not img_path: continue

            concepts = [0] * 7
            masks = [0.0] * 7
            label = 1 if row.get('dx') == 'mel' else 0
            self.samples.append((img_path, concepts, masks, label))

    def __getitem__(self, idx):
        img_path, concepts, masks, label = self.samples[idx]
        try:
            img = Image.open(img_path).convert("RGB")
        except FileNotFoundError:
            img = Image.new('RGB', (224, 224))
        if self.transforms:
            img = self.transforms(img)
        return (
            img,
            torch.tensor(concepts, dtype=torch.long),
            torch.tensor(masks, dtype=torch.float32),
            torch.tensor(label, dtype=torch.float32)
        )

Overwriting /content/drive/MyDrive/XAI_Cloud_Run/data_loaders.py


In [9]:
!python train.py

Streaming output truncated to the last 5000 lines.
                                                              0.000             
                                                              train_loss_epoch: 
Epoch 3/49 ╸━━━━━━━━━━━━━━━━ 5/101 0:00:03 • 0:00:43 2.26it/s v_num: 1.000      
                                                              train_loss_step:  
                                                              0.431 val_loss:   
                                                              0.835 val_auc:    
                                                              0.000 val_recall: 
                                                              0.000             
                                                              train_loss_epoch: 
Epoch 3/49 ━╺━━━━━━━━━━━━━━━ 6/101 0:00:03 • 0:00:37 2.61it/s v_num: 1.000      
                                                              train_loss_step:  
                                                          

In [10]:
%%writefile /content/drive/MyDrive/XAI_Cloud_Run/train.py
import os
import torch
import pandas as pd
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms
from sklearn.model_selection import train_test_split

from config import Config
from data_loaders import Derm7ptDataset
from models import GatedHybridCBM
from engine import CBM_System

def main():
    pl.seed_everything(42)
    print("Initializing Cloud Training Sequence...")

    DATA_DIR = "/content/drive/MyDrive/XAI_Cloud_Run/derm7pt"
    META_CSV = os.path.join(DATA_DIR, "meta", "meta.csv")
    IMAGE_DIR = os.path.join(DATA_DIR, "images")
    CHECKPOINT_DIR = "/content/drive/MyDrive/XAI_Cloud_Run/cloud_checkpoints/"

    full_df = pd.read_csv(META_CSV)

    # ---------------------------------------------------------
    # THE SANITATION BLOCK
    # ---------------------------------------------------------
    print("🧹 Sanitizing dirty medical text...")
    clean_diag = full_df['diagnosis'].astype(str).str.lower().str.strip()
    binary_labels = clean_diag.str.contains('melanoma').astype(int)

    total_melanomas = binary_labels.sum()
    print(f"Found exactly {total_melanomas} Melanoma instances in dataset.")

    # Stratification is now mathematically safe
    train_df, val_df = train_test_split(
        full_df, test_size=0.2, stratify=binary_labels, random_state=42
    )

    print("Calculating Class Weights...")
    # Update weight calculation to use the sanitized logic
    train_clean_diag = train_df['diagnosis'].astype(str).str.lower().str.strip()
    is_melanoma = train_clean_diag.str.contains('melanoma')

    count_melanoma = is_melanoma.sum()
    count_other = len(train_df) - count_melanoma

    weight_melanoma = 1.0 / count_melanoma if count_melanoma > 0 else 1.0
    weight_other = 1.0 / count_other if count_other > 0 else 1.0

    sample_weights = [weight_melanoma if val else weight_other for val in is_melanoma]

    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True
    )

    basic_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor()
    ])

    train_dataset = Derm7ptDataset(metadata_df=train_df, image_dir=IMAGE_DIR, transforms=basic_transform)
    val_dataset = Derm7ptDataset(metadata_df=val_df, image_dir=IMAGE_DIR, transforms=basic_transform)

    train_loader = DataLoader(train_dataset, batch_size=8, sampler=sampler, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)

    print("Building GatedHybridCBM Architecture...")
    model = GatedHybridCBM(
        concept_dims=Config.get_active_concept_dims(),
        backbone=Config.BACKBONE,
        feat_proj_dim=Config.FEAT_PROJ_DIM,
        hidden=Config.HIDDEN_DIM,
        dropout=Config.DROPOUT
    )
    lightning_system = CBM_System(model, Config)

    checkpoint_callback = ModelCheckpoint(
        dirpath=CHECKPOINT_DIR,
        filename="gated-cbm-{epoch:02d}-{val_auc:.3f}",
        monitor="val_auc",
        mode="max",
        save_top_k=1,
    )

    early_stop_callback = EarlyStopping(
        monitor="val_auc",
        min_delta=0.005,
        patience=7,
        verbose=True,
        mode="max"
    )

    logger = TensorBoardLogger("/content/drive/MyDrive/XAI_Cloud_Run/tb_logs", name="GatedHybridCBM_Derm7pt")

    trainer = pl.Trainer(
        max_epochs=50,
        accelerator="auto",
        devices="auto",
        logger=logger,
        callbacks=[checkpoint_callback, early_stop_callback],
        precision="16-mixed",
        log_every_n_steps=10
    )

    print("🔥 Launching Training Sequence...")
    trainer.fit(lightning_system, train_dataloaders=train_loader, val_dataloaders=val_loader)
    print(f"Training Complete. Best model saved at: {checkpoint_callback.best_model_path}")

if __name__ == "__main__":
    main()

Overwriting /content/drive/MyDrive/XAI_Cloud_Run/train.py


In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
%%writefile /content/drive/MyDrive/XAI_Cloud_Run/data_loaders.py
import os
import pandas as pd
import numpy as np
import torch
from PIL import Image
from base_dataset import BaseConceptDataset

class Derm7ptDataset(BaseConceptDataset):
    def __init__(self, metadata_df, image_dir, transforms=None):
        super().__init__(metadata_df, transforms)
        self.image_dir = image_dir
        self.concept_cols = [
            "pigment_network", "streaks", "pigmentation",
            "regression_structures", "dots_and_globules",
            "blue_whitish_veil", "vascular_structures"
        ]
        self._prepare_samples()

    def _parse_concept(self, val, col_idx):
        if pd.isna(val) or val == -1 or str(val).strip() == '-1' or str(val).lower() == 'unknown':
            return 0, 0.0
        if isinstance(val, (int, float)):
            return int(val), 1.0

        val_str = str(val).lower().strip()
        if val_str in ['absent', 'none', 'false', '0']: return 0, 1.0
        if val_str in ['present', 'typical', 'regular', 'true', '1']: return 1, 1.0
        if val_str in ['atypical', 'irregular', '2']:
            return 2, 1.0 if col_idx == 5 else (1, 1.0)
        return 1, 1.0

    def _prepare_samples(self):
        for _, row in self.df.iterrows():
            concepts, masks = [], []
            for i, col in enumerate(self.concept_cols):
                val = row.get(col, np.nan)
                c_val, c_mask = self._parse_concept(val, i)
                concepts.append(c_val)
                masks.append(c_mask)

            label = 1 if str(row.get('diagnosis')).lower().strip() == 'melanoma' else 0

            # EXTRACT BOTH IMAGES: Doubles the physical dataset size
            derm_img = row.get('derm')
            clinic_img = row.get('clinic')

            if pd.notna(derm_img) and str(derm_img).strip() != '':
                self.samples.append((os.path.join(self.image_dir, str(derm_img)), concepts, masks, label))
            if pd.notna(clinic_img) and str(clinic_img).strip() != '':
                self.samples.append((os.path.join(self.image_dir, str(clinic_img)), concepts, masks, label))

    def __getitem__(self, idx):
        img_path, concepts, masks, label = self.samples[idx]
        try:
            if os.path.isdir(img_path): raise IsADirectoryError()
            img = Image.open(img_path).convert("RGB")
        except (FileNotFoundError, IsADirectoryError):
            img = Image.new('RGB', (224, 224))

        if self.transforms:
            img = self.transforms(img)

        return (
            img,
            torch.tensor(concepts, dtype=torch.long),
            torch.tensor(masks, dtype=torch.float32),
            torch.tensor(label, dtype=torch.float32)
        )

Overwriting /content/drive/MyDrive/XAI_Cloud_Run/data_loaders.py


In [11]:
!python train.py

Streaming output truncated to the last 5000 lines.
                                                               1.055 val_auc:   
                                                               0.344 val_recall:
                                                               0.000            
                                                               train_loss_epoch:
                                                               0.407            
Epoch 3/49 ━━━━━━━━━━━━━━━━ 101/101 0:00:20 • 0:00:00 5.08it/s v_num: 2.000     
                                                               train_loss_step: 
                                                               0.226 val_loss:  
                                                               1.055 val_auc:   
                                                               0.344 val_recall:
                                                               0.000            
                                                          

In [11]:
%%writefile /content/drive/MyDrive/XAI_Cloud_Run/data_loaders.py
import os
import pandas as pd
import numpy as np
import torch
from PIL import Image
from base_dataset import BaseConceptDataset

class Derm7ptDataset(BaseConceptDataset):
    def __init__(self, metadata_df, image_dir, transforms=None):
        super().__init__(metadata_df, transforms)
        self.image_dir = image_dir
        self.concept_cols = [
            "pigment_network", "streaks", "pigmentation",
            "regression_structures", "dots_and_globules",
            "blue_whitish_veil", "vascular_structures"
        ]
        self._prepare_samples()

    def _parse_concept(self, val, col_idx):
        if pd.isna(val) or val == -1 or str(val).strip() == '-1' or str(val).lower() == 'unknown':
            return 0, 0.0
        if isinstance(val, (int, float)):
            return int(val), 1.0

        val_str = str(val).lower().strip()
        if val_str in ['absent', 'none', 'false', '0']:
            return 0, 1.0
        if val_str in ['present', 'typical', 'regular', 'true', '1']:
            return 1, 1.0
        if val_str in ['atypical', 'irregular', '2']:
            if col_idx == 5:
                return 2, 1.0
            else:
                return 1, 1.0
        return 1, 1.0

    def _prepare_samples(self):
        for _, row in self.df.iterrows():
            concepts, masks = [], []
            for i, col in enumerate(self.concept_cols):
                val = row.get(col, np.nan)
                c_val, c_mask = self._parse_concept(val, i)
                concepts.append(c_val)
                masks.append(c_mask)

            label = 1 if str(row.get('diagnosis')).lower().strip() == 'melanoma' else 0

            # EXTRACT BOTH IMAGES: Doubles the physical dataset size
            derm_img = row.get('derm')
            clinic_img = row.get('clinic')

            if pd.notna(derm_img) and str(derm_img).strip() != '':
                self.samples.append((os.path.join(self.image_dir, str(derm_img)), concepts, masks, label))
            if pd.notna(clinic_img) and str(clinic_img).strip() != '':
                self.samples.append((os.path.join(self.image_dir, str(clinic_img)), concepts, masks, label))

    def __getitem__(self, idx):
        img_path, concepts, masks, label = self.samples[idx]
        try:
            if os.path.isdir(img_path): raise IsADirectoryError()
            img = Image.open(img_path).convert("RGB")
        except (FileNotFoundError, IsADirectoryError):
            img = Image.new('RGB', (224, 224))

        if self.transforms:
            img = self.transforms(img)

        return (
            img,
            torch.tensor(concepts, dtype=torch.long),
            torch.tensor(masks, dtype=torch.float32),
            torch.tensor(label, dtype=torch.float32)
        )

Overwriting /content/drive/MyDrive/XAI_Cloud_Run/data_loaders.py


In [12]:
%%writefile /content/drive/MyDrive/XAI_Cloud_Run/train.py
import os
import torch
import pandas as pd
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms
from sklearn.model_selection import train_test_split

from config import Config
from data_loaders import Derm7ptDataset
from models import GatedHybridCBM
from engine import CBM_System

def main():
    pl.seed_everything(42)
    print("🚀 Initializing Phase 1.5: The Concept Forge...")

    DATA_DIR = "/content/drive/MyDrive/XAI_Cloud_Run/derm7pt"
    META_CSV = os.path.join(DATA_DIR, "meta", "meta.csv")
    IMAGE_DIR = os.path.join(DATA_DIR, "images")
    CHECKPOINT_DIR = "/content/drive/MyDrive/XAI_Cloud_Run/cloud_checkpoints/"

    full_df = pd.read_csv(META_CSV)

    clean_diag = full_df['diagnosis'].astype(str).str.lower().str.strip()
    binary_labels = clean_diag.str.contains('melanoma').astype(int)

    train_df, val_df = train_test_split(
        full_df, test_size=0.2, stratify=binary_labels, random_state=42
    )

    print("⚖️ Calculating Class Weights...")
    train_clean_diag = train_df['diagnosis'].astype(str).str.lower().str.strip()
    is_melanoma = train_clean_diag.str.contains('melanoma')
    count_melanoma = is_melanoma.sum()
    count_other = len(train_df) - count_melanoma

    weight_melanoma = 1.0 / count_melanoma if count_melanoma > 0 else 1.0
    weight_other = 1.0 / count_other if count_other > 0 else 1.0
    sample_weights = [weight_melanoma if val else weight_other for val in is_melanoma]

    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True
    )

    # ---------------------------------------------------------
    # AGGRESSIVE AUGMENTATION
    # ---------------------------------------------------------
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=45),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.ToTensor()
    ])

    # Validation must remain pristine
    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor()
    ])

    # Instantiate datasets with new dual-image loading and transforms
    train_dataset = Derm7ptDataset(metadata_df=train_df, image_dir=IMAGE_DIR, transforms=train_transform)
    val_dataset = Derm7ptDataset(metadata_df=val_df, image_dir=IMAGE_DIR, transforms=val_transform)

    train_loader = DataLoader(train_dataset, batch_size=8, sampler=sampler, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)

    print("🧠 Building GatedHybridCBM Architecture...")
    model = GatedHybridCBM(
        concept_dims=Config.get_active_concept_dims(),
        backbone=Config.BACKBONE,
        feat_proj_dim=Config.FEAT_PROJ_DIM,
        hidden=Config.HIDDEN_DIM,
        dropout=Config.DROPOUT
    )
    lightning_system = CBM_System(model, Config)

    checkpoint_callback = ModelCheckpoint(
        dirpath=CHECKPOINT_DIR,
        filename="forged-cbm-{epoch:02d}-{val_auc:.3f}",
        monitor="val_auc",
        mode="max",
        save_top_k=1,
    )

    early_stop_callback = EarlyStopping(
        monitor="val_auc",
        min_delta=0.005,
        patience=10, # Increased patience because augmentation slows down convergence
        verbose=True,
        mode="max"
    )

    logger = TensorBoardLogger("/content/drive/MyDrive/XAI_Cloud_Run/tb_logs", name="GatedHybridCBM_Forged")

    trainer = pl.Trainer(
        max_epochs=50,
        accelerator="auto",
        devices="auto",
        logger=logger,
        callbacks=[checkpoint_callback, early_stop_callback],
        precision="16-mixed",
        log_every_n_steps=10
    )

    print("🔥 Launching The Concept Forge...")
    trainer.fit(lightning_system, train_dataloaders=train_loader, val_dataloaders=val_loader)
    print(f"✅ Training Complete. Best model saved at: {checkpoint_callback.best_model_path}")

if __name__ == "__main__":
    main()

Overwriting /content/drive/MyDrive/XAI_Cloud_Run/train.py


In [13]:
# 1. Move the terminal into your workspace
%cd /content/drive/MyDrive/XAI_Cloud_Run/

# 2. Fire the engine
!python train.py

Streaming output truncated to the last 5000 lines.
                                                               1.405 val_auc:   
                                                               0.797 val_recall:
                                                               0.000            
                                                               train_loss_epoch:
                                                               0.260            
Validation  ━━━━━━━━━━━━━╺━━ 43/51   0:00:05 •        7.88it/s                  
Epoch 12/49 ━━━━━━━━━━━━━━━━ 101/101 0:00:24 •        4.22it/s v_num: 1.000     
                                     0:00:00                   train_loss_step: 
                                                               0.202 val_loss:  
                                                               1.405 val_auc:   
                                                               0.797 val_recall:
                                                          